# Civic issue classifier: train in Colab
Trains a YOLO image classifier for: **pothole, garbage, water leakage, broken streetlight**.
Run the cells top to bottom. **Before you start:** Runtime > Change runtime type > **T4 GPU**.

Only image datasets work here. Kaggle datasets loaded as tables (CSV sensor data) cannot train a photo model;
step 3 tells you which of yours contain images.

## 1. Setup

In [ ]:
!nvidia-smi -L || echo "NO GPU: Runtime > Change runtime type > T4 GPU, then re-run"
!pip -q install ultralytics kagglehub pyyaml scikit-learn

In [ ]:
import os
REPO = "https://github.com/deepshikhathakur020105/AI_civic_managemnet_system.git"
if not os.path.exists("civic"):
    # If the repo is private this fails: upload the ml/ folder to Colab instead.
    !git clone -q -b ml-dataset-yolo $REPO civic
%cd civic/ml

## 2. Log in to Kaggle and download
Create a token at kaggle.com > Settings > API > Create New Token. `login()` asks for your username and token.

In [ ]:
import kagglehub
kagglehub.login()

In [ ]:
# key -> Kaggle dataset slug ("owner/name" from the dataset URL).
# Replace the pothole placeholder: search kaggle.com for "pothole" and pick one that has IMAGES.
DATASETS = {
    "garbage":      "sumn2u/garbage-classification-v2",
    "water_leak":   "ziya07/water-leak-dataset",
    "streetlight":  "vizeno/street-light-fault-prediction-dataset",
    "pothole":      "<owner>/<pothole-dataset-slug>",
}

paths = {}
for key, slug in DATASETS.items():
    if slug.startswith("<"):
        print(f"SKIP {key}: fill in the Kaggle slug first")
        continue
    try:
        paths[key] = kagglehub.dataset_download(slug)
        print(f"OK   {key}: {paths[key]}")
    except Exception as e:
        print(f"FAIL {key} ({slug}): {e}")

## 3. Inspect: does each dataset really contain images?

In [ ]:
from collections import Counter
from pathlib import Path
IMG = {".jpg", ".jpeg", ".png"}

def inspect(root):
    root = Path(root)
    files = [p for p in root.rglob("*") if p.is_file()]
    imgs = [p for p in files if p.suffix.lower() in IMG]
    table = [p for p in files if p.suffix.lower() in {".csv", ".xlsx", ".parquet"}]
    return imgs, table

for key, root in paths.items():
    imgs, table = inspect(root)
    print(f"\n=== {key}: {len(imgs)} images, {len(table)} table files ===")
    if not imgs:
        print("  NO IMAGES -> cannot train a photo classifier from this dataset.")
        for t in table[:3]:
            print("  table:", t.name)
        continue
    for folder, n in Counter(p.parent.name for p in imgs).most_common(25):
        print(f"  folder '{folder}': {n} images")
    ymls = list(Path(root).rglob("data.yaml"))
    if ymls:
        print("  YOLO-format dataset. data.yaml:", ymls[0])

## 4. Map datasets to your classes
Edit this cell using the folder names printed above. Each entry is `folder name -> your class`.
Datasets with no images are simply left out, and your model will only learn the classes that have images.

Caveat for garbage: `garbage-classification-v2` is close-up photos of waste items, not piles of rubbish on a street.
It teaches the model "garbage-like objects", so test on your own street photos before trusting it.

In [ ]:
import yaml

# Folder-per-class image datasets: {dataset key: {folder name: your class}}
FOLDER_MAP = {
    "garbage": {"trash": "garbage", "plastic": "garbage", "paper": "garbage", "cardboard": "garbage",
                "glass": "garbage", "metal": "garbage", "biological": "garbage"},
    # "streetlight": {"broken": "broken_streetlight"},     # only if it has image folders
    # "water_leak":  {"leak": "water_leakage"},            # only if it has image folders
}

# YOLO-format datasets (have data.yaml): {dataset key: (your class, source class names to keep or None for all)}
YOLO_SOURCES = {
    # "pothole": ("pothole", None),
}

sources, classes = [], []
for key, mapping in FOLDER_MAP.items():
    if key not in paths:
        continue
    for folder, cls in mapping.items():
        dirs = [d for d in Path(paths[key]).rglob(folder) if d.is_dir()]
        if not dirs:
            print(f"WARNING: folder '{folder}' not found in {key}")
        for i, d in enumerate(dirs):
            sources.append({"name": f"{key}_{folder}_{i}", "type": "folders",
                            "path": str(d.parent), "map": {folder: cls}})
            if cls not in classes:
                classes.append(cls)

for key, (cls, keep) in YOLO_SOURCES.items():
    if key not in paths:
        continue
    ymls = list(Path(paths[key]).rglob("data.yaml"))
    if not ymls:
        print(f"WARNING: no data.yaml in {key}; use FOLDER_MAP instead")
        continue
    names = yaml.safe_load(open(ymls[0]))["names"]
    names = list(names.values()) if isinstance(names, dict) else list(names)
    print(f"{key}: source classes = {names}")
    sources.append({"name": key, "type": "yolo", "path": str(paths[key]), "names": names,
                    "map": {n: cls for n in names if keep is None or n in keep}})
    if cls not in classes:
        classes.append(cls)

assert sources, "No usable sources: fix the mapping above."
yaml.safe_dump({"classes": classes, "sources": sources}, open("datasets.yaml", "w"))
print("Classes that will be trained:", classes)

## 5. Merge, de-duplicate and split (80/10/10)

In [ ]:
!python prepare_dataset.py --config datasets.yaml --out data/civic --max-per-class 2000

## 6. Train
`yolo11n-cls.pt` is a small pretrained model, so this only fine-tunes it.
Watch `top1_acc` on the validation set; if it stops improving, training is done.

In [ ]:
!python train.py --data data/civic --epochs 30 --device 0

## 7. Evaluate on the held-out test set

In [ ]:
!python evaluate.py --data data/civic

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
cm = pd.read_csv("reports/confusion_matrix.csv", index_col=0)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm.values, cmap="Blues")
ax.set_xticks(range(len(cm.columns))); ax.set_xticklabels(cm.columns, rotation=45, ha="right")
ax.set_yticks(range(len(cm.index))); ax.set_yticklabels(cm.index)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm.values[i, j], ha="center", va="center")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); fig.colorbar(im)
plt.tight_layout(); plt.savefig("reports/confusion_matrix.png", dpi=150); plt.show()

## 8. Try your own photo
Use a real street photo, not one from the dataset. This is the test that matters.

In [ ]:
from google.colab import files
from ultralytics import YOLO
model = YOLO("../backend/models/best.pt")
for name in files.upload():
    r = model(name, verbose=False)[0]
    top = r.probs.top5
    print(name, "->", [(r.names[i], round(float(r.probs.data[i]), 2)) for i in top[:3]])

## 9. Save the model
Put `best.pt` into `backend/models/` in your project. Do not commit it to git.

In [ ]:
from google.colab import files
files.download("../backend/models/best.pt")
files.download("reports/metrics.json")
files.download("reports/confusion_matrix.png")